<a href="https://colab.research.google.com/github/Srikrishna69420/AutoAugment-Implementation-and-Theory/blob/main/AutoAugment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np
from PIL import Image, ImageEnhance, ImageOps

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import Subset

SEED = 420

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#AutoAugment

In this colab, I will be implementing the AutoAugment paper. There will be three major parts in this colab. The first part will include the main part of the paper, which is the idea of policies. The second part will include the LSTM, where the best policy will be found. Finally, the last part will include the training of the child NN and application of the above two parts.

#Policies
In this part, I will implement policies used in the paper. I will define the different augmentation techniques. I will be implementing 15 out of 16, and will be excluding Sample Pairing due to complications.

##Augmentation
In this part, I implement the 15 augmentation techniques.

###ShearX

In [ ]:
def ShearX(img, magnitude):

    return img.transform(
        img.size,
        Image.AFFINE,
        (1, magnitude, 0, 0, 1, 0),
        Image.BICUBIC)

###ShearY

In [ ]:
def ShearY(img, magnitude):

    return img.transform(
        img.size,
        Image.AFFINE,
        (1, 0, 0, magnitude, 1, 0),
        Image.BICUBIC)

###TranslateX

In [ ]:
def TranslateX(img, magnitude):

    magnitude = magnitude * img.size[0]

    return img.transform(
        img.size,
        Image.AFFINE,
        (1, 0, magnitude, 0, 1, 0))

###TranslateY

In [ ]:
def TranslateY(img, magnitude):

    magnitude = magnitude * img.size[1]

    return img.transform(
        img.size,
        Image.AFFINE,
        (1, 0, 0, 0, 1, magnitude))

###Rotate

In [ ]:
def Rotate(img, magnitude):

    return img.rotate(magnitude)

###AutoContrast

In [ ]:
def AutoContrast(img, magnitude=None):

    return ImageOps.autocontrast(img)

###Equalize

In [ ]:
def Equalize(img, magnitude=None):

    return ImageOps.equalize(img)

###Invert

In [ ]:
def Invert(img, magnitude=None):

    return ImageOps.invert(img)

###Solarize

In [ ]:
def Solarize(img, magnitude):

    return ImageOps.solarize(img, magnitude)

###Posterize

In [ ]:
def Posterize(img, magnitude):

    magnitude = max(1, int(magnitude))

    return ImageOps.posterize(img, magnitude)

###Contrast

In [ ]:
def Contrast(img, magnitude):

    return ImageEnhance.Contrast(img).enhance(magnitude)

###Colour

In [ ]:
def Colour(img, magnitude):

    return ImageEnhance.Color(img).enhance(magnitude)

###Brightness

In [ ]:
def Brightness(img, magnitude):

    return ImageEnhance.Brightness(img).enhance(magnitude)

###Sharpness

In [ ]:
def Sharpness(img, magnitude):

    return ImageEnhance.Sharpness(img).enhance(magnitude)

###Cutout

In [ ]:
def Cutout(img, magnitude):

    img = np.array(img).copy()
    h, w, c = img.shape
    mask_size = int(magnitude)

    y = np.random.randint(h)
    x = np.random.randint(w)

    y1 = np.clip(y - mask_size // 2,0,h)
    y2 = np.clip(y + mask_size // 2,0,h)
    x1 = np.clip(x - mask_size // 2,0,w)
    x2 = np.clip(x + mask_size // 2,0,w)

    img[y1:y2, x1:x2, :] = 0

    return Image.fromarray(img)

##Search Space
In this part, we define the search space for policies. From this search space, we try to find the best policy.

###Operations

In [ ]:
OPS = [
    "ShearX",
    "ShearY",
    "TranslateX",
    "TranslateY",
    "Rotate",
    "AutoContrast",
    "Equalize",
    "Invert",
    "Solarize",
    "Posterize",
    "Contrast",
    "Colour",
    "Brightness",
    "Sharpness",
    "Cutout"]

###Operation Maps

In [ ]:
NAME_TO_OP = {
    "ShearX": ShearX,
    "ShearY": ShearY,
    "TranslateX": TranslateX,
    "TranslateY": TranslateY,
    "Rotate": Rotate,
    "AutoContrast": AutoContrast,
    "Equalize": Equalize,
    "Invert": Invert,
    "Solarize": Solarize,
    "Posterize": Posterize,
    "Contrast": Contrast,
    "Colour": Colour,
    "Brightness": Brightness,
    "Sharpness": Sharpness,
    "Cutout": Cutout}

###Ranges for Magnitude

In [ ]:
RANGES = {
    "ShearX": np.linspace(0, 0.3, 10),
    "ShearY": np.linspace(0, 0.3, 10),
    "TranslateX": np.linspace(0, 0.45, 10),
    "TranslateY": np.linspace(0, 0.45, 10),
    "Rotate": np.linspace(0, 30, 10),
    "Solarize": np.linspace(256, 0, 10),
    "Posterize": np.round(np.linspace(8, 4, 10),0).astype(int),
    "Contrast": np.linspace(0.1, 1.9, 10),
    "Colour": np.linspace(0.1, 1.9, 10),
    "Brightness": np.linspace(0.1, 1.9, 10),
    "Sharpness": np.linspace(0.1, 1.9, 10),
    "Cutout": np.linspace(4, 20, 10),
    "AutoContrast": [0] * 10,
    "Equalize": [0] * 10,
    "Invert": [0] * 10}

##AutoAugment Policy
In this part, I will implement the policy. We generate a random number belonging to [0,1). We further check if this is greater than the defined probability. If yes, we proceed, otherwise we dont. After this, we define the magnitude using the above function. Also, if the operation is one of the below, we also need to define it for negative magnitudes.

In [ ]:
class AutoAugmentPolicy:

    def __init__(self, sub_policy):
        self.sub_policy = sub_policy

    def __call__(self, img):

      subpolicy = random.choice(self.sub_policy)

      for op_name, prob, mag_idx in subpolicy:

        if random.random() < prob:

            magnitude = RANGES[op_name][mag_idx]

            if op_name in [
                "ShearX",
                "ShearY",
                "TranslateX",
                "TranslateY",
                "Rotate"]:

                if random.random() > 0.5:
                    magnitude = -magnitude

            img = NAME_TO_OP[op_name](
                img,
                magnitude)

      return img


With this, I conclude the first major part of this paper. I defined all the parameters, magnitude, actual magnitude and the autoaugment policy.

#LSTM
In this part, I define the rest of the paper, including the RL controller,

##RL Controller
In this part, I will define the RL Controller. This is a crucial part of the paper. Firstly, we define a dummy input token. This is converted into an embedding in 100-dimensional space. Follwoing this, it is fed into LSTM to try and outputs hidden representation. This is fed into 3 seperate prediction layers, called heads. The first head predicts which type of augmentation to use. The second head predicts the probability of applying this augmentation. The final head predicts the magnitude of augmentation. This is fed into the LSTM again. This allows for sequential policy generation, rather than random generation. It is also looped over twice, given the original design.


In [ ]:
class Controller(nn.Module):

    def __init__(self):

        super().__init__()
        self.hidden_size = 100
        self.embedding = nn.Embedding(100,self.hidden_size)

        self.lstm = nn.LSTM(
            input_size=self.hidden_size,
            hidden_size=self.hidden_size,
            batch_first=True)

        self.op_head = nn.Linear(
            self.hidden_size,
            len(OPS))

        self.prob_head = nn.Linear(
            self.hidden_size,
            11)

        self.mag_head = nn.Linear(
            self.hidden_size,
            10)

    def sample_policy(self):

        inputs = torch.zeros(
            (1, 1),
            dtype=torch.long).to(device)

        hidden = None
        log_probs = []
        policy = []

        for _ in range(5):
          subpolicy = []
          for _ in range(2):
            embedded = self.embedding(inputs)
            output,hidden = self.lstm(embedded,hidden)
            output = output[:, -1]

            op_logits = self.op_head(output)
            op_dist = torch.distributions.Categorical(logits=op_logits)

            op_idx = op_dist.sample()
            log_probs.append(op_dist.log_prob(op_idx))
            op_name = OPS[op_idx.item()]

            prob_logits = self.prob_head(output)
            prob_dist = torch.distributions.Categorical(logits=prob_logits)
            prob_idx = prob_dist.sample()
            log_probs.append(prob_dist.log_prob(prob_idx))
            probability = (prob_idx.item() / 10.0)

            mag_logits = self.mag_head(output)
            mag_dist = torch.distributions.Categorical(logits=mag_logits)
            mag_idx = mag_dist.sample()

            log_probs.append(mag_dist.log_prob(mag_idx))
            magnitude = mag_idx.item()

            subpolicy.append((
                op_name,
                probability,
                magnitude))

            inputs = op_idx.unsqueeze(0)
          policy.append(subpolicy)

        total_log_prob = torch.stack(log_probs).sum()

        return policy, total_log_prob

##Child NN
In this, I define the child network, which will be used to generate validation accuracy for a given policy. This accuracy will be used as reward for RL controller. We use the WideResNet for this.

In [ ]:
class WideBasic(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride,
        dropout_rate=0.3):

        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=True)
        self.dropout = nn.Dropout(
            p=dropout_rate)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=True)
        self.shortcut = nn.Sequential()

        if (stride != 1 or in_channels != out_channels):

            self.shortcut = nn.Sequential(
                nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=True))

    def forward(self, x):

        out = self.dropout(self.conv1(torch.relu(self.bn1(x))))
        out = self.conv2(torch.relu(self.bn2(out)))
        out += self.shortcut(x)

        return out

class WideResNet(nn.Module):

    def __init__(
        self,
        depth=28,
        widen_factor=10,
        dropout_rate=0.3,
        num_classes=10):

        super().__init__()
        self.in_channels = 16
        assert ((depth - 4) % 6 == 0)
        n = (depth - 4) // 6
        k = widen_factor
        nStages = [16,16 * k, 32 * k,64 * k]

        self.conv1 = nn.Conv2d(
            3,
            nStages[0],
            kernel_size=3,
            stride=1,
            padding=1,
            bias=True)

        self.layer1 = self._wide_layer(
            WideBasic,
            nStages[1],
            n, stride=1,
            dropout_rate=dropout_rate)

        self.layer2 = self._wide_layer(
            WideBasic,
            nStages[2],
            n, stride=2,
            dropout_rate=dropout_rate)

        self.layer3 = self._wide_layer(
            WideBasic,
            nStages[3],
            n, stride=2,
            dropout_rate=dropout_rate)

        self.bn1 = nn.BatchNorm2d(nStages[3])

        self.linear = nn.Linear(
            nStages[3],
            num_classes)

    def _wide_layer(
        self,
        block,
        out_channels,
        num_blocks,
        stride,
        dropout_rate):

        strides = [stride] + [1] * (num_blocks - 1)
        layers = []

        for stride in strides:

            layers.append(
                block(
                    self.in_channels,
                    out_channels,
                    stride,
                    dropout_rate))

            self.in_channels = out_channels

        return nn.Sequential(*layers)

    def forward(self, x):

        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = torch.relu(self.bn1(out))
        out = torch.nn.functional.avg_pool2d(out,8)
        out = out.view(out.size(0),-1)
        out = self.linear(out)

        return out

##Datasets and Policy Datasets
In this part, I load and prepare the data. I also define the necessary subset, consisting of 1000 images which will be used for the autoaugment itself.

In [ ]:
full_train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True)

indices = list(range(4000))    #only the first 1000 images, due to computational constraints, original paper had 4000 images

train_subset = Subset(
    full_train_dataset,
    indices)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
         (0.2023, 0.1994, 0.2010))])

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform)


class PolicyDataset(Dataset):

    def __init__(self, subset, transform):

        self.subset = subset
        self.transform = transform

    def __len__(self):

        return len(self.subset)

    def __getitem__(self, idx):

        img, label = self.subset[idx]
        img = self.transform(img)

        return img, label

100%|██████████| 170M/170M [00:03<00:00, 45.5MB/s]


#Training
In this last part, I will define the training and the policy search part of the paper, which will also be the most computationally expensive part.

##Training of Child NN
Here, we train a NN with training dataset augmented as the policy. The accuracy will be used as reward for RL controller. I used only 20 epochs because of computational cost.

In [ ]:
def train_child(policy):

    train_transform = transforms.Compose([
        AutoAugmentPolicy(policy),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4914, 0.4822, 0.4465),
             (0.2023, 0.1994, 0.2010))])

    train_dataset = PolicyDataset(
        train_subset,
        train_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=16,
        shuffle=True)

    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False)

    model = WideResNet(
    depth=28,
    widen_factor=10,
    dropout_rate=0.3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001)


    model.train()

    for epoch in range(120):

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = criterion(
                outputs,
                labels)

            loss.backward()
            optimizer.step()


    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
      for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    return accuracy

##RL Search and Results
Finally, I implemented the RL Search, which looks thorugh the search space to try and find the best policy. We see if the reward, which is the validation accuracy for a policy, is better than the best reward. If it is, then we update the best policy and best reward. Once the program ends, the best policy and best reward are printed. We also use concept of baseline, and its defined as the average of rewards, more specifically moving averages. We use this to check if our policy is generally better than the average via advantage. I also limited the number of policies to 50.

In [ ]:
controller = Controller().to(device)
controller_optimizer = optim.Adam(
    controller.parameters(),
    lr=0.00035)

baseline = None
NUM_POLICIES = 10
best_policy = None
best_reward = 0

for iteration in range(NUM_POLICIES):

    policy, log_prob = controller.sample_policy()

    print(f"\nPolicy {iteration + 1}")
    print(policy)

    reward = train_child(policy)
    print(f"Validation Accuracy: {reward:.4f}")

    if baseline is None:
      baseline = reward
    else:
      baseline = (0.95 * baseline + 0.05 * reward)

    advantage = reward - baseline


    loss = -log_prob * advantage
    controller_optimizer.zero_grad()
    loss.backward()
    controller_optimizer.step()

    if reward > best_reward:

        best_reward = reward
        best_policy = policy

print("\nBest Policy:")
print(best_policy)
print(f"Best Validation Accuracy: {best_reward:.4f}")


Policy 1
[[('Brightness', 0.0, 2), ('ShearX', 0.8, 9)], [('AutoContrast', 0.5, 1), ('ShearY', 0.7, 2)], [('TranslateX', 0.3, 9), ('Invert', 0.3, 4)], [('Invert', 0.4, 2), ('AutoContrast', 0.3, 5)], [('Colour', 0.5, 5), ('AutoContrast', 0.2, 6)]]


KeyboardInterrupt: 

# Observations
In the end, I have implemnted a lightweight version of the paper. They have trained it for about 5000 GPU hours, where GPU hours are estimated for an NVIDIA Tesla P100. I dont have the resources or the time for the full implementation, therefore I have made the following changes for the lightweight version:
1. Number of policies limited to 50 (paper had 15,000)
2. Epoch limited to 20 (paper had 120)
3. Subset size limited to 1000 (paper had 4000)

It is interesting to note that the best accuracy I achieved was only 43%. This is mainly because of extreme reduction in number of policies, which hinders the program from exploring more options (total options include (15x11x10)^10 approximately 1.49x10^32). Also the paper specifically suggested the epochs to be 80-100 to fully benifit from the sub-policies, however my version includes only 20.

(Number of Policies, Epoch, Subset size) - (Best Validation Accuracy)
1. (50, 20, 1000) - (42.97%)
2. (100, 60, 1000) - (52.72%)
3. (3, 120, 4000) - (70.62%)